# 2. Ingest – Ficha técnica de vehículos (`car_features.csv`)
**Capa:** raw ➜ bronze  |  **Destino:** `vehiculos.bronze.car_features`

- Lectura del CSV desde el container `raw` con **schema explícito**.
- El valor `N/A` de la fuente se interpreta como nulo (`nullValue`).
- Renombrado de columnas a *snake_case* (la fuente mezcla mayúsculas, espacios y `_`).
- Columnas de auditoría: `source_file` e `ingestion_date`.

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, current_timestamp

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "vehiculos")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsjgl99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/car_features.csv"

### Schema explícito

In [0]:
car_features_schema = StructType(fields=[
    StructField("Make", StringType(), False),
    StructField("Model", StringType(), False),
    StructField("Year", IntegerType(), True),
    StructField("Engine_Fuel_Type", StringType(), True),
    StructField("Engine_HP", IntegerType(), True),
    StructField("Engine_Cylinders", IntegerType(), True),
    StructField("Transmission_Type", StringType(), True),
    StructField("Driven_Wheels", StringType(), True),
    StructField("Number_of_Doors", IntegerType(), True),
    StructField("Market_Category", StringType(), True),
    StructField("Vehicle_Size", StringType(), True),
    StructField("Vehicle_Style", StringType(), True),
    StructField("Highway_MPG", IntegerType(), True),
    StructField("City_MPG", IntegerType(), True),
    StructField("Popularity", IntegerType(), True),
    StructField("MSRP", IntegerType(), True)
])

In [0]:
df_car_features = spark.read \
    .option("header", True) \
    .option("nullValue", "N/A") \
    .schema(car_features_schema) \
    .csv(ruta)

In [0]:
display(df_car_features.limit(10))

### Renombrado y columnas de auditoría

In [0]:
car_features_renamed_df = df_car_features \
    .withColumnRenamed("Make", "make") \
    .withColumnRenamed("Model", "model") \
    .withColumnRenamed("Year", "year") \
    .withColumnRenamed("Engine_Fuel_Type", "engine_fuel_type") \
    .withColumnRenamed("Engine_HP", "engine_hp") \
    .withColumnRenamed("Engine_Cylinders", "engine_cylinders") \
    .withColumnRenamed("Transmission_Type", "transmission_type") \
    .withColumnRenamed("Driven_Wheels", "driven_wheels") \
    .withColumnRenamed("Number_of_Doors", "number_of_doors") \
    .withColumnRenamed("Market_Category", "market_category") \
    .withColumnRenamed("Vehicle_Size", "vehicle_size") \
    .withColumnRenamed("Vehicle_Style", "vehicle_style") \
    .withColumnRenamed("Highway_MPG", "highway_mpg") \
    .withColumnRenamed("City_MPG", "city_mpg") \
    .withColumnRenamed("Popularity", "popularity") \
    .withColumnRenamed("MSRP", "msrp")

In [0]:
car_features_final_df = car_features_renamed_df \
    .withColumn("source_file", col("_metadata.file_name")) \
    .withColumn("ingestion_date", current_timestamp())

### Validación previa a la carga

In [0]:
total = car_features_final_df.count()
duplicados = total - car_features_final_df.drop("ingestion_date").distinct().count()
modelos = car_features_final_df.select("make", "model").distinct().count()

print(f"Registros leídos       : {total}")
print(f"Filas duplicadas       : {duplicados}  (se depuran en silver)")
print(f"Combinaciones make/model: {modelos}")

display(car_features_final_df.select(
    *[ (car_features_final_df[c].isNull().cast("int")).alias(c) for c in
       ["engine_fuel_type", "engine_hp", "engine_cylinders", "number_of_doors", "market_category"] ]
).groupBy().sum())

### Escritura en bronze
Se respeta el orden de columnas de la tabla porque `insertInto` inserta por posición.

In [0]:
columnas_tabla = spark.table(f"{catalogo}.{esquema}.car_features").columns

car_features_final_df.select(*columnas_tabla) \
    .write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.car_features")

In [0]:
display(spark.table(f"{catalogo}.{esquema}.car_features")
        .groupBy("make").count().orderBy(col("count").desc()))